In [3]:
import numpy as np
import io

import matplotlib.pyplot as plt

import sys
import os

# Assuming /AutomationModule is in the root directory of your project
sys.path.append(os.path.abspath(r'H:\phd stuff\tidy3d'))

from AutomationModule import * 

import AutomationModule as AM

In [4]:
file_path = r"H:\phd stuff\tidy3d\structures\End2EndFiles\filtered\lsu_0p8_filter.txt"

In [5]:
def sample_schulz(z,f, n,threshold: float = None):
    """Samples from a Schulz distribution.
    Parameters:
    z (float): The shape parameter of the Schulz distribution.
    f (float): The mean of the distribution.
    n (int): The number of samples to generate.
    threshold (float): The maximum value to cap the samples at.
    Returns:
    np.ndarray: An array of sampled values from the Schulz distribution.
    """
    shape = z + 1
    s_mean =np.sqrt(f/(1+1/(z+1))) 
    scale = s_mean / (z + 1)
    values = np.random.gamma(shape=shape, scale=scale, size=n)
    if threshold is not None:
        values[values >= threshold] = threshold
    return values

In [6]:
def create_permittivity_grid_penlike(
    rod_endpoints,
    grid_size=128,
    minor_radius=0.1,        # circular radius in the pre-warp space (becomes ellipse after z-warp)
    aspect_ratio=None,        # global z scale factor s; ellipses have major/minor = s (projected)
    aspect_ratio_hole=None,        # global z scale factor s; ellipses have major/minor = s (projected)
    permittivity=3.42**2, #Si
    hole_minor_radius=0.0,   # inner circular radius in pre-warp space (same eccentricity after warp)
    box_size=14.3,
    *,
    progress_every=20,
    dinamic_radius=False,
    distribution_type:str="normal",
    schulz_params:dict={"z":0,"f":0,"threshold":None},
    sigma_inner=None,
    sigma_outer=None,
    mean_inner=None,
    mean_outer=None,
    create_hole=False,
    use_radius_array=False,
    b_list=None

):
    """
    Build hollow 'pen-like' rods by:
      (1) creating RIGHT CIRCULAR cylinders in an unwarped space (x, y, z'),
      (2) applying a GLOBAL z warp: z = s * z'.

    IMPORTANT:
      - 'rod_endpoints' are given in FINAL (post-warp) world coords.
      - We unwarp them internally: z' = z / s.
      - Because the warp is GLOBAL, both shell and hole share the SAME ellipticity.
        You can set different radii (thickness), but not different ellipse ARs.

    Geometry:
      - Axis unit vector and clamping 0<=t<=L are computed in UNWARPED space.
      - Membership test is purely circular in UNWARPED cross-sections.
      - All ellipses are aligned with the projection of +z into each rod cross-plane.
        For rods parallel to z, the cross-section remains circular (expected).

    Returns:
      grid: (G,G,G) float32 array, ones background with 'permittivity' written inside rods.
    """
    rods = np.asarray(rod_endpoints, dtype=np.float32)  # scale to match original design
    G = int(grid_size)
    grid = np.ones((G, G, G), dtype=np.float32)
    solid_grid = np.ones((G, G, G), dtype=np.float32)

    dx = float(box_size) / G
    
    grid_coords = (np.arange(G, dtype=np.float32) + 0.5) * dx - np.float32(box_size / 2.0)

    # ----- radii setup (ensure arrays always exist) -----
    if not dinamic_radius:
        b_array  = np.full(len(rods), float(minor_radius), dtype=np.float32)
        b_h_array = np.full(len(rods), float(hole_minor_radius), dtype=np.float32)
    elif use_radius_array:
        if b_list is None:
            raise ValueError("b_list must be provided when use_radius_array=True.")
        if len(b_list) != len(rods):
            raise ValueError("b_list must have the same length as rod_endpoints.")
        b_array  = np.asarray(b_list, dtype=np.float32)
        if distribution_type == "normal":
            b_h_array = np.random.normal(loc=mean_inner, scale=sigma_inner, size=len(rods)).astype(np.float32)
        else: 
            b_h_array = sample_schulz(z=schulz_params["z"],f=schulz_params["f"],n=len(rods),threshold=schulz_params["threshold"])*b_array
    else:
        b_array  = np.random.normal(loc=mean_outer, scale=sigma_outer, size=len(rods)).astype(np.float32)
        if distribution_type == "normal":
            b_h_array = np.random.normal(loc=mean_inner, scale=sigma_inner, size=len(rods)).astype(np.float32)
        else:
            b_h_array = sample_schulz(z=schulz_params["z"],f=schulz_params["f"],n=len(rods),threshold=schulz_params["threshold"])*b_array

    s = float(aspect_ratio) if aspect_ratio is not None else 1.0
    s_in = float(aspect_ratio_hole) if aspect_ratio_hole is not None else 1.0
    k_in  = s / s_in  # inner boundary z' scaling in UNWARPED space

    def idx_range_for_world(min_w, max_w, pad):
        lo = min_w - pad; hi = max_w + pad
        i0 = int(np.searchsorted(grid_coords, lo, side='left'))
        i1 = int(np.searchsorted(grid_coords, hi, side='right') - 1)
        i0 = max(i0, 0); i1 = min(i1, G - 1)
        if i1 < i0:
            mid = 0.5 * (min_w + max_w)
            i0 = i1 = max(min(int(np.searchsorted(grid_coords, mid, side='left')), G - 1), 0)
        return i0, i1

    for i_rod, rod in enumerate(rods):
        if progress_every and (i_rod % progress_every == 0):
            print(f"[postwarp] rod {i_rod} / {len(rods)}")

        b = b_array[i_rod]
        b_h = b_h_array[i_rod]
            
        use_hole = (b_h > 0.0) and (b_h < b)
        
        # pad in WORLD space: radius along directions with z-component can grow by up to s
        r_pad_world = b * max(1.0, s) + dx
        # FINAL (world) endpoints
        p1w = rod[:3].astype(np.float32)
        p2w = rod[3:].astype(np.float32)

        # UNWARP endpoints (z' = z / s) to build circular cylinder there
        p1u = p1w.copy(); p1u[2] = p1w[2] / s
        p2u = p2w.copy(); p2u[2] = p2w[2] / s

        vu = p2u - p1u
        L2u = float(np.dot(vu, vu))
        if L2u <= 0.0:
            continue
        Lu = float(np.sqrt(L2u))
        nu = vu / Lu  # axis in UNWARPED space

        # WORLD AABB expanded
        xmin, xmax = float(min(p1w[0], p2w[0])), float(max(p1w[0], p2w[0]))
        ymin, ymax = float(min(p1w[1], p2w[1])), float(max(p1w[1], p2w[1]))
        zmin, zmax = float(min(p1w[2], p2w[2])), float(max(p1w[2], p2w[2]))
        ix0, ix1 = idx_range_for_world(xmin, xmax, r_pad_world)
        iy0, iy1 = idx_range_for_world(ymin, ymax, r_pad_world)
        iz0, iz1 = idx_range_for_world(zmin, zmax, r_pad_world)

        xs = grid_coords[ix0:ix1+1]
        ys = grid_coords[iy0:iy1+1]
        zs = grid_coords[iz0:iz1+1]
        X, Y, Z = np.meshgrid(xs, ys, zs, indexing='ij')

        # Map WORLD coords to UNWARPED coords (x, y, z') with z' = z / s
        Zu = Z / s

        # Vector from p1 in UNWARPED space
        RXu = X - p1u[0]
        RYu = Y - p1u[1]
        RZu = Zu - p1u[2]

        # Axial coordinate and clamping in UNWARPED space
        tu = RXu * nu[0] + RYu * nu[1] + RZu * nu[2]
        mask_len = (tu >= 0.0) & (tu <= Lu)

        # Perpendicular distance in UNWARPED space (circular test)
        rX = RXu - tu * nu[0]
        rY = RYu - tu * nu[1]
        rZ = RZu - tu * nu[2]


        r2 = rX**2 + rY**2 + rZ**2
        outer_ok = r2 <= (b**2)


        if use_hole and create_hole:
            r2_inner = rX**2 + rY**2 + (k_in * rZ)**2
            inner_ok = r2_inner <= (b_h**2)
            final_mask = mask_len & outer_ok & (~inner_ok)
            solid_mask = mask_len & outer_ok
        else:
            final_mask = mask_len & outer_ok
            solid_mask = mask_len & outer_ok

        if np.any(final_mask):
            sub = grid[ix0:ix1+1, iy0:iy1+1, iz0:iz1+1]
            sub[final_mask] = permittivity
            grid[ix0:ix1+1, iy0:iy1+1, iz0:iz1+1] = sub



        if np.any(solid_mask):
            sub_solid = solid_grid[ix0:ix1+1, iy0:iy1+1, iz0:iz1+1]
            sub_solid[solid_mask] = permittivity
            solid_grid[ix0:ix1+1, iy0:iy1+1, iz0:iz1+1] = sub_solid


    def get_ff(grid):
        grid_ff = np.copy(grid)
        grid_ff[grid_ff==1] = 0
        grid_ff[grid_ff>0] = 1
        return grid_ff.mean()

    #Filling fraction 
    material_ff = get_ff(grid)
    original_ff = get_ff(solid_grid)

    holes_ff = 1-material_ff/original_ff

    print({"rod":material_ff, "original":original_ff, "holes":holes_ff})

    return grid,b_array,[material_ff, original_ff, holes_ff]


In [7]:
# Load the data from the string
rod_endpoints = np.loadtxt(file_path)

len(rod_endpoints)

1653

In [8]:
#Creating constant radius array for all rods
GRID_SIZE = 512  # Resolution of the grid (e.g., 64, 128, 256)
MINOR_RADIUS = 0.2252 # The smaller radius of the elliptical cross-section
# HOLE_MINOR_RADIUS = 0 # The smaller radius of the elliptical cross-section
ASPEC = 2.5

PERM = 3.4**2

# Generate the permittivity grid
permittivity_grid,b_array,ff = create_permittivity_grid_penlike(rod_endpoints, grid_size=GRID_SIZE, minor_radius=MINOR_RADIUS, aspect_ratio=ASPEC, progress_every=None, permittivity=PERM,box_size=11.44)

{'rod': 0.21720098, 'original': 0.21720098, 'holes': 0.0}


In [ ]:
directory = rf"./structures/n_{np.sqrt(PERM):.4f}/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_cut_dist_corrected"
os.makedirs(directory, exist_ok=True)
AM.create_hdf5_from_dict({"epsilon":permittivity_grid},rf"{directory}/n_{np.sqrt(PERM):.4f}_ffh_{ff[2]:.4f}_ffr_{ff[0]:.4f}_fforiginal_{ff[1]:.4f}_z_0.h5")

In [11]:

AM.create_hdf5_from_dict({"rads":b_array},rf"{directory}/rads_ff_{ff[0]:.4f}.h5") #Got radius array for the generated structure, can be used to create new structures with the same radius distribution but different random seeds (i.e., different spatial arrangements of rods)

In [ ]:
GRID_SIZE = 512  # Resolution of the grid (e.g., 64, 128, 256)
schulz_params={"f":0.235,"z":100,"threshold":None}
PERM = 3.4**2

for i in range(5): #Create 5 different structures with the same radius distribution but different spatial arrangements of rods
    rads_minor = AM.read_hdf5_as_dict(r"./structures/n_rads_ff_0.2172.h5")['rads'] #Loaded radius distibution for structure with ff ~ 20% (same as our structure)
    permittivity_grid,b_array,ff = create_permittivity_grid_penlike(rod_endpoints, grid_size=GRID_SIZE, aspect_ratio=2.5, permittivity=PERM,box_size=11.44,
                                                         aspect_ratio_hole=2.5, progress_every=False, 
                                                         dinamic_radius=True, 
                                                         create_hole=True,
                                                         use_radius_array=True,
                                                         b_list=rads_minor,
                                                         distribution_type="schulz",
                                                         schulz_params=schulz_params
                                                         )

    directory =  rf"./structures/n_{np.sqrt(PERM):.2f}/Schulz/AR Hole 2.5/ff_0p2172/ffh_0.225_cut_dist_corrected/z_{schulz_params["z"]}"
    os.makedirs(directory, exist_ok=True)
    AM.create_hdf5_from_dict({"epsilon":permittivity_grid},rf"{directory}/n_{np.sqrt(PERM):.2f}_ffh_{ff[2]:.4f}_ffr_{ff[0]:.4f}_fforiginal_{ff[1]:.4f}_z_{schulz_params["z"]}_{i}.h5")

{'rod': 0.16856603, 'original': 0.21720098, 'holes': 0.22391676902770996}
{'rod': 0.16855513, 'original': 0.21720098, 'holes': 0.22396701574325562}
{'rod': 0.16932476, 'original': 0.21720098, 'holes': 0.2204236388206482}
{'rod': 0.16864821, 'original': 0.21720098, 'holes': 0.22353845834732056}
{'rod': 0.16899286, 'original': 0.21720098, 'holes': 0.2219516634941101}
